In [1]:
import os
import time
import gc
import numpy as np
import pandas as pd

DATASETS = {
    "SIFT1M": {
        "dimension": 128,
        "docs": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/sift/sift_docs.csv",
        "queries": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/sift/sift_queries.csv",
        "output": "/home/salemmohammed/Projects/research-projects/HyVec/results/raw/GTTest/sift_ground_truth_k1000.npz",
        "attribute": "rand_int",
        "K": 1000,
    },
    "PAPER": {
        "dimension": 200,
        "docs": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/paper/paper_docs.csv",
        "queries": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/paper/paper_queries.csv",
        "output": "/home/salemmohammed/Projects/research-projects/HyVec/results/raw/GTTest/paper_ground_truth_k1000.npz",
        "attribute": "rand_int",
        "K": 1000,
    },
    "BIER": {
        "dimension": 1024,
        "docs": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/beir/beir_docs.csv",
        "queries": "/home/salemmohammed/Projects/research-projects/HyVec/data/hnsw/beir/beir_queries.csv",
        "output": "/home/salemmohammed/Projects/research-projects/HyVec/results/raw/GTTest/bier_ground_truth_k1000.npz",
        "attribute": "source",
        "K": 1000,
    },
}

In [2]:
def parse_vector(vector_string):
    return np.fromstring(str(vector_string),sep=",",dtype=np.float32)

In [3]:
def filtered_brute_force_search(
    database_vectors,
    database_attributes,
    database_ids,
    query_vector,
    query_attribute,
    k,
    verbose=False,
):
    """
    Exact filtered brute-force search for one query.
    """

    # Find rows whose metadata matches the query metadata.
    matching_positions = np.flatnonzero(database_attributes == query_attribute)

    # Prepare empty results.
    result_ids = np.full(k,-1,dtype=np.int64)

    result_distances = np.full(k,np.inf,dtype=np.float32)

    # No database vector satisfies the predicate.
    if matching_positions.size == 0:
        return result_ids, result_distances, 0

    # Retrieve only metadata-compatible vectors.
    matching_vectors = database_vectors[matching_positions]

    # Calculate exact squared Euclidean distances.
    differences = matching_vectors - query_vector

    distances = np.sum( differences * differences, axis=1, dtype=np.float32 )

    # Return at most k results.
    number_to_return = min(k,matching_positions.size)

    # Select the closest k vectors.
    top_local_positions = np.argpartition( distances, number_to_return - 1)[:number_to_return]

    # Sort the selected results by distance.
    top_local_positions = top_local_positions[np.argsort(distances[top_local_positions])]

    # Convert local filtered positions to database positions.
    top_database_positions = matching_positions[top_local_positions]

    # Store result IDs and distances.

    result_ids[:number_to_return] = database_ids[top_database_positions]

    result_distances[:number_to_return] = distances[top_local_positions]
    if verbose:
        print(f"\nMatching count: {matching_positions.size}")
        print(f"Top {number_to_return} result IDs:       {result_ids[:number_to_return]}")
        print(f"Top {number_to_return} result distances: {result_distances[:number_to_return]}")

    return ( result_ids, result_distances, matching_positions.size )

In [4]:
# example

# --- Fake tiny "database" ---
database_vectors = np.array([
    [1.0, 1.0],   # id 100
    [2.0, 2.0],   # id 101
    [9.0, 9.0],   # id 102
    [1.5, 1.0],   # id 103
    [0.0, 0.0],   # id 104
    [3.0, 3.0],   # id 105
])
database_attributes = np.array(["A", "B", "A", "A", "B", "A"])
database_ids = np.array([100, 101, 102, 103, 104, 105])

# --- Fake query ---
query_vector = np.array([1.0, 1.0])
query_attribute = "A"
k = 3

# --- Run the function ---

ids, dists, count = filtered_brute_force_search(
    database_vectors, database_attributes, database_ids, query_vector, query_attribute, k, True
)
print("matching count:", count)
print("result ids:     ", ids)
print("result distances:", dists)


Matching count: 4
Top 3 result IDs:       [100 103 105]
Top 3 result distances: [0.   0.25 8.  ]
matching count: 4
result ids:      [100 103 105]
result distances: [0.   0.25 8.  ]


In [5]:
def load_dataset(dataset_name,config):

    print("\n" + "=" * 70)
    print(f"Loading {dataset_name}")
    print("=" * 70)

    docs_df = pd.read_csv(config["docs"], sep=";")

    queries_df = pd.read_csv(config["queries"],sep=";")

    print(f"Documents : {len(docs_df):,}")
    print(f"Queries   : {len(queries_df):,}")

    return docs_df, queries_df

In [6]:
def prepare_dataset_arrays(
    dataset_name,
    docs_df,
    queries_df,
    expected_dimension,
    attribute_column,
    verbose=False,
    verbose_n=5,
):
    """
    Parse embeddings and extract metadata arrays.
    """

    print(f"Parsing {dataset_name} document vectors...")

    document_vector_list = [parse_vector(value) for value in docs_df["embedding"]]

    # Validate every document vector.
    for row_number, vector in enumerate( document_vector_list):
        if vector.size != expected_dimension:
            raise ValueError(
                f"{dataset_name} document row "
                f"{row_number} has dimension "
                f"{vector.size}; expected "
                f"{expected_dimension}."
            )

    database_vectors = np.vstack(document_vector_list).astype(np.float32,copy=False)

    del document_vector_list

    print(f"Parsing {dataset_name} query vectors...")

    query_vector_list = [parse_vector(value) for value in queries_df["embedding"]]
    
    if verbose:
        print(f"\n--- Sample of {verbose_n} query rows (before/after parsing) ---")
        for i in range(min(verbose_n, len(query_vector_list))):
            raw = queries_df["embedding"].iloc[i]
            parsed = query_vector_list[i]
            print(f"[query {i}] raw (first 60 chars): {str(raw)[:60]}...")
            print(f"[query {i}] parsed shape: {parsed.shape}, first 5 values: {parsed[:5]}")

    # Validate every query vector.
    for row_number, vector in enumerate(query_vector_list):
        if vector.size != expected_dimension:
            raise ValueError(
                f"{dataset_name} query row "
                f"{row_number} has dimension "
                f"{vector.size}; expected "
                f"{expected_dimension}."
            )

    query_vectors = np.vstack(query_vector_list).astype(np.float32,copy=False)

    del query_vector_list

    # Extract metadata.
    database_attributes = docs_df[ attribute_column].to_numpy()

    query_attributes = queries_df[attribute_column].to_numpy()

    # Create numeric IDs.
    database_ids = np.arange(database_vectors.shape[0],dtype=np.int64)

    query_ids = np.arange(query_vectors.shape[0], dtype=np.int64)
    
    if verbose:
        print(f"\n--- Sample of {verbose_n} attributes ---")
        print(f"Database attributes[:{verbose_n}]: {database_attributes[:verbose_n]}")
        print(f"Query    attributes[:{verbose_n}]: {query_attributes[:verbose_n]}")
        print(f"Database attribute dtype: {database_attributes.dtype}")
        print(f"Query    attribute dtype: {query_attributes.dtype}")

    print("Database shape:", database_vectors.shape)
    print("Query shape:", query_vectors.shape)

    return (
        database_vectors,
        query_vectors,
        database_ids,
        query_ids,
        database_attributes,
        query_attributes,
    )

In [7]:
def process_dataset(dataset_name,config):
    """
    Generate exact filtered ground truth
    for one dataset.
    """

    # Load CSV files.
    docs_df, queries_df = load_dataset(dataset_name, config)

    # Convert data into NumPy arrays.
    database_vectors, query_vectors, database_ids, query_ids, database_attributes, query_attributes = prepare_dataset_arrays(
        dataset_name=dataset_name,
        docs_df=docs_df,
        queries_df=queries_df,
        expected_dimension=config["dimension"],
        attribute_column=config["attribute"],
        )

    number_of_queries = query_vectors.shape[0]
    k = config["K"]

    # Prepare output matrices.
    all_neighbor_ids = np.full((number_of_queries, k), -1, dtype=np.int64 )

    all_neighbor_distances = np.full((number_of_queries, k),np.inf,dtype=np.float32)

    matching_counts = np.zeros(number_of_queries,dtype=np.int64)

    print(f"Running filtered brute-force search " f"for {dataset_name}...")

    start_time = time.perf_counter()

    # Run the function once for every query.
    for query_index in range(number_of_queries):

        (
            neighbor_ids,
            neighbor_distances,
            matching_count,
        ) = filtered_brute_force_search(
            database_vectors=database_vectors,
            database_attributes=database_attributes,
            database_ids=database_ids,
            query_vector=query_vectors[query_index],
            query_attribute=query_attributes[query_index],
            k=k,
        )

        all_neighbor_ids[query_index] = neighbor_ids

        all_neighbor_distances[query_index] = neighbor_distances

        matching_counts[query_index] = matching_count

        if ((query_index + 1) % 100 == 0 or query_index + 1 == number_of_queries):
            print( f"Completed {query_index + 1:,} of {number_of_queries:,} queries "
            f"| matching_count={matching_count:,} "
            f"| nearest_distance={neighbor_distances[0]:.4f} "
            f"| top5_distances={neighbor_distances[:5]}"
            )
    total_time = (time.perf_counter() - start_time)

    # Create output directory.
    output_path = config["output"]

    output_directory = os.path.dirname(output_path)

    if output_directory:
        os.makedirs(output_directory,exist_ok=True)

    # Save ground truth.
    # Save ground truth.
    np.savez_compressed(
        output_path,
        query_ids=query_ids,
        neighbor_ids=all_neighbor_ids,
        neighbor_distances=all_neighbor_distances,
        query_attributes=query_attributes,
        matching_counts=matching_counts,
        k=np.array(k, dtype=np.int64),
        dataset_name=np.array(dataset_name),
        total_time_seconds=np.array(total_time, dtype=np.float64),
        database_size=np.array(database_vectors.shape[0], dtype=np.int64),
        dimension=np.array(config["dimension"], dtype=np.int64),
    )

    # Derived performance metrics.
    average_latency_ms = (total_time / number_of_queries) * 1000.0
    qps = number_of_queries / total_time

    # Derived filter diagnostics.
    average_matching_vectors = float(np.mean(matching_counts))
    average_selectivity_percent = (average_matching_vectors / database_vectors.shape[0]) * 100.0

    print(f"\nFinished {dataset_name}")
    print("Saved to:", output_path)
    print(f"Total time: {total_time:.3f} seconds")
    print(f"Average latency: {average_latency_ms:.3f} ms")
    print(f"QPS: {qps:.3f}")
    print(f"Average matching vectors: {average_matching_vectors:.1f}")
    print(f"Average selectivity: {average_selectivity_percent:.3f}%")

    result = {
        "Method": "Filtered Brute Force",
        "Dataset": dataset_name,
        "Recall@K": 1.0,
        "K": k,
        "Database Vectors": database_vectors.shape[0],
        "Queries": number_of_queries,
        "Dimension": config["dimension"],
        "Total Time (s)": total_time,
        "Average Latency (ms)": average_latency_ms,
        "QPS": qps,
        "Average Matching Vectors": average_matching_vectors,
        "Average Selectivity (%)": average_selectivity_percent,
        "Output Path": output_path,
    }

    # Release large arrays before the next dataset.
    del (
        docs_df,
        queries_df,
        database_vectors,
        query_vectors,
        database_ids,
        query_ids,
        database_attributes,
        query_attributes,
        all_neighbor_ids,
        all_neighbor_distances,
        matching_counts,
    )

    gc.collect()

    return result

In [8]:
dataset_name = "SIFT1M"  # change to "PAPER" or "BIER" as needed
config = DATASETS[dataset_name]

result = process_dataset(dataset_name, config)

results_df = pd.DataFrame([result])

print("\nFinished.")

display(results_df)



Loading SIFT1M
Documents : 1,000,000
Queries   : 10,000
Parsing SIFT1M document vectors...
Parsing SIFT1M query vectors...
Database shape: (1000000, 128)
Query shape: (10000, 128)
Running filtered brute-force search for SIFT1M...
Completed 100 of 10,000 queries | matching_count=83,772 | nearest_distance=69616.0000 | top5_distances=[69616. 75481. 78734. 79940. 80517.]
Completed 200 of 10,000 queries | matching_count=82,681 | nearest_distance=46554.0000 | top5_distances=[46554. 49981. 58203. 60050. 60956.]
Completed 300 of 10,000 queries | matching_count=82,681 | nearest_distance=40184.0000 | top5_distances=[40184. 43611. 46510. 50803. 51341.]
Completed 400 of 10,000 queries | matching_count=83,406 | nearest_distance=28970.0000 | top5_distances=[28970. 43602. 48021. 49075. 50066.]
Completed 500 of 10,000 queries | matching_count=83,287 | nearest_distance=63319.0000 | top5_distances=[63319. 71530. 74466. 74935. 75631.]
Completed 600 of 10,000 queries | matching_count=83,652 | nearest_dis

,Method,Dataset,Recall@K,K,Database Vectors,Queries,Dimension,Total Time (s),Average Latency (ms),QPS,Average Matching Vectors,Average Selectivity (%),Output Path
0,Filtered Brute Force,SIFT1M,1.0,1000,1000000,10000,128,159.317984,15.931798,62.767553,83337.6289,8.333763,/home/salemmohammed/Projects/research-projects...


In [ ]:
dataset_name = "PAPER"  # change to "PAPER" or "BIER" as needed
config = DATASETS[dataset_name]

result = process_dataset(dataset_name, config)

results_df = pd.DataFrame([result])

print("\nFinished.")

display(results_df)



Loading PAPER
Documents : 2,029,997
Queries   : 10,000
Parsing PAPER document vectors...


In [ ]:
dataset_name = "BIER"  # change to "PAPER" or "BIER" as needed
config = DATASETS[dataset_name]

result = process_dataset(dataset_name, config)

results_df = pd.DataFrame([result])

print("\nFinished.")

display(results_df)